## Load Aligned Sequence Data

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORT_DIR = PROJECT_ROOT / "reports"

sequence_config = json.loads(
    (REPORT_DIR / "sequence_preparation_config.json").read_text(
        encoding="utf-8"
    )
)

SEQUENCE_DIR = PROJECT_ROOT / sequence_config["sequence_directory"]

history_dir = SEQUENCE_DIR / "historical_train"
later_dir = SEQUENCE_DIR / "later_evaluation"

history_sequences = np.load(
    history_dir / "sequences.npy",
    mmap_mode="r",
)

history_transactions = np.load(
    history_dir / "transaction_inputs.npy",
    mmap_mode="r",
)

history_labels = np.load(
    history_dir / "labels.npy",
    mmap_mode="r",
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

assert history_sequences.shape[0] == len(history_labels)
assert history_transactions.shape[0] == len(history_labels)
assert history_sequences.shape[1:] == (30, 9)
assert np.isin(history_labels, [0, 1]).all()

print("Device:", DEVICE)
print("Historical sequence shape:", history_sequences.shape)
print("Historical transaction shape:", history_transactions.shape)

Device: mps
Historical sequence shape: (882127, 30, 9)
Historical transaction shape: (882127, 7)


## Create an Internal Historical Validation Split

In [2]:
all_history_indices = np.arange(len(history_labels))

train_indices, valid_indices = train_test_split(
    all_history_indices,
    test_size=0.2,
    stratify=history_labels,
    random_state=SEED,
)

train_indices = np.sort(train_indices)
valid_indices = np.sort(valid_indices)

assert np.intersect1d(train_indices, valid_indices).size == 0
assert len(train_indices) + len(valid_indices) == len(history_labels)

print("Internal training users:", len(train_indices))
print("Internal validation users:", len(valid_indices))
print(
    "Training churn rate:",
    f"{history_labels[train_indices].mean():.2%}",
)
print(
    "Validation churn rate:",
    f"{history_labels[valid_indices].mean():.2%}",
)

Internal training users: 705701
Internal validation users: 176426
Training churn rate: 9.66%
Validation churn rate: 9.66%


## Fit Transaction Preprocessing on Internal Training Users Only

In [3]:
transaction_preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True),
    ),
    ("scaler", StandardScaler()),
])

transaction_preprocessor.fit(
    history_transactions[train_indices]
)

history_static = transaction_preprocessor.transform(
    history_transactions
).astype(np.float32)

assert np.isfinite(history_static).all()

static_feature_names = (
    transaction_preprocessor.get_feature_names_out(
        sequence_config["transaction_columns"]
    ).tolist()
)

print("Processed transaction input shape:", history_static.shape)
print("Processed transaction features:", static_feature_names)

Processed transaction input shape: (882127, 10)
Processed transaction features: ['transaction_rows_90d', 'transaction_days_90d', 'cancel_rows_90d', 'auto_renew_share_90d', 'avg_plan_days_90d', 'days_since_last_transaction_90d', 'has_transaction_90d', 'missingindicator_auto_renew_share_90d', 'missingindicator_avg_plan_days_90d', 'missingindicator_days_since_last_transaction_90d']


## Fit Sequence Normalization on Valid Training Observations

In [4]:
def fit_sequence_normalization(sequences, indices, chunk_size=4096):
    sums = np.zeros(7, dtype=np.float64)
    squared_sums = np.zeros(7, dtype=np.float64)
    observation_counts = np.zeros(7, dtype=np.int64)

    for start in range(0, len(indices), chunk_size):
        selected_indices = indices[start:start + chunk_size]
        block = np.asarray(sequences[selected_indices])

        assert np.isfinite(block).all()

        values = block[:, :, :7].astype(np.float64)

        log_present = block[:, :, 7] == 1
        duration_valid = block[:, :, 8] == 1

        assert not (duration_valid & ~log_present).any()

        masks = np.broadcast_to(
            log_present[:, :, None],
            values.shape,
        ).copy()

        masks[:, :, 6] = duration_valid

        masked_values = np.where(masks, values, 0.0)

        sums += masked_values.sum(axis=(0, 1))
        squared_sums += (masked_values ** 2).sum(axis=(0, 1))
        observation_counts += masks.sum(axis=(0, 1))

    safe_counts = np.maximum(observation_counts, 1)
    means = sums / safe_counts

    variances = np.maximum(
        squared_sums / safe_counts - means ** 2,
        0.0,
    )

    standard_deviations = np.sqrt(variances)
    standard_deviations[
        standard_deviations < 1e-6
    ] = 1.0

    return (
        means.astype(np.float32),
        standard_deviations.astype(np.float32),
        observation_counts,
    )

sequence_means, sequence_stds, sequence_observations = (
    fit_sequence_normalization(
        history_sequences,
        train_indices,
    )
)

normalization_summary = pd.DataFrame({
    "channel": sequence_config["sequence_channels"][:7],
    "training_observations": sequence_observations,
    "mean": sequence_means,
    "std": sequence_stds,
})

normalization_summary

,channel,training_observations,mean,std
0,log1p_num_25,9058806,1.326469,1.108256
1,log1p_num_50,9058806,0.610696,0.711879
2,log1p_num_75,9058806,0.479263,0.593161
3,log1p_num_985,9058806,0.504884,0.615684
4,log1p_num_100,9058806,2.808356,1.175393
5,log1p_num_unq,9058806,2.921139,1.044511
6,log1p_valid_seconds,9056881,8.321076,1.285998


## Read and Normalize Sequence Batches

In [5]:
def iter_batches(
    sequences,
    static_inputs,
    labels,
    indices,
    means,
    stds,
    batch_size=1024,
    shuffle=False,
    seed=42,
):
    order = np.asarray(indices).copy()

    if shuffle:
        np.random.default_rng(seed).shuffle(order)

    for start in range(0, len(order), batch_size):
        selected = order[start:start + batch_size]

        sequence_batch = np.array(
            sequences[selected],
            dtype=np.float32,
            copy=True,
        )

        log_present = sequence_batch[:, :, 7].copy()
        duration_valid = sequence_batch[:, :, 8].copy()

        sequence_batch[:, :, :7] = (
            sequence_batch[:, :, :7] - means
        ) / stds

        sequence_batch[:, :, :6] *= log_present[:, :, None]
        sequence_batch[:, :, 6] *= duration_valid

        static_batch = np.asarray(
            static_inputs[selected],
            dtype=np.float32,
        )

        label_batch = np.asarray(
            labels[selected],
            dtype=np.float32,
        )

        assert np.isfinite(sequence_batch).all()
        assert np.isfinite(static_batch).all()

        yield (
            torch.from_numpy(sequence_batch),
            torch.from_numpy(static_batch),
            torch.from_numpy(label_batch),
        )

## Define a GRU Model with Transaction Inputs

In [6]:
from torch import nn

class ChurnGRU(nn.Module):
    def __init__(
        self,
        sequence_channels,
        static_features,
        hidden_size=32,
        dropout=0.1,
        prior_probability=0.1,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=sequence_channels,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )

        self.static_branch = nn.Sequential(
            nn.Linear(static_features, 16),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size + 16, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

        initial_bias = np.log(
            prior_probability / (1 - prior_probability)
        )

        nn.init.constant_(self.head[-1].bias, float(initial_bias))

    def forward(self, sequences, static_inputs):
        _, hidden = self.gru(sequences)

        sequence_embedding = hidden[-1]
        static_embedding = self.static_branch(static_inputs)

        combined = torch.cat(
            [sequence_embedding, static_embedding],
            dim=1,
        )

        return self.head(combined).squeeze(-1)

model_config = {
    "sequence_channels": history_sequences.shape[2],
    "static_features": history_static.shape[1],
    "hidden_size": 32,
    "dropout": 0.1,
    "prior_probability": float(
        history_labels[train_indices].mean()
    ),
}

torch.manual_seed(SEED)
gru_model = ChurnGRU(**model_config).to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in gru_model.parameters()
    if parameter.requires_grad
)

print(gru_model)
print("Trainable parameters:", parameter_count)

ChurnGRU(
  (gru): GRU(9, 32, batch_first=True)
  (static_branch): Sequential(
    (0): Linear(in_features=10, out_features=16, bias=True)
    (1): ReLU()
  )
  (head): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)
Trainable parameters: 5905


In [7]:
smoke_sequences, smoke_static, smoke_labels = next(
    iter_batches(
        history_sequences,
        history_static,
        history_labels,
        train_indices[:32],
        sequence_means,
        sequence_stds,
        batch_size=32,
    )
)

missing_days = smoke_sequences[:, :, 7] == 0
invalid_duration = smoke_sequences[:, :, 8] == 0

assert (
    smoke_sequences[:, :, :7][missing_days] == 0
).all()

assert (
    smoke_sequences[:, :, 6][invalid_duration] == 0
).all()

gru_model.train()
gru_model.zero_grad(set_to_none=True)

logits = gru_model(
    smoke_sequences.to(DEVICE),
    smoke_static.to(DEVICE),
)

assert logits.shape == (len(smoke_labels),)
assert torch.isfinite(logits).all().item()

criterion = nn.BCEWithLogitsLoss()

smoke_loss = criterion(
    logits,
    smoke_labels.to(DEVICE),
)

assert torch.isfinite(smoke_loss).item()

smoke_loss.backward()

for parameter in gru_model.parameters():
    if parameter.grad is not None:
        assert torch.isfinite(parameter.grad).all().item()

gru_model.zero_grad(set_to_none=True)

print("Forward and backward checks passed.")
print("Smoke-test loss:", smoke_loss.item())

Forward and backward checks passed.
Smoke-test loss: 0.44720518589019775


## Train with Internal Historical Early Stopping

Select the training epoch using internal validation Log Loss.
The later evaluation cohort is not used for early stopping.

In [8]:
from time import perf_counter
from scipy.special import expit

from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

BATCH_SIZE = 1024
LEARNING_RATE = 0.001
MAX_EPOCHS = 12
PATIENCE = 3

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

internal_checkpoint = MODEL_DIR / "gru_internal_best.pt"

def train_one_epoch(
    model,
    optimizer,
    indices,
    static_inputs,
    means,
    stds,
    epoch,
):
    model.train()

    total_loss = 0.0
    total_users = 0

    batches = iter_batches(
        history_sequences,
        static_inputs,
        history_labels,
        indices,
        means,
        stds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED + epoch,
    )

    for batch_number, (sequences, static, labels) in enumerate(
        batches, start=1
    ):
        sequences = sequences.to(DEVICE)
        static = static.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(sequences, static)
        loss = criterion(logits, labels)

        assert torch.isfinite(loss).item(), "Non-finite training loss."

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
        )

        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_users += len(labels)

        if batch_number % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"users={total_users:,} | "
                f"mean loss={total_loss / total_users:.5f}",
                flush=True,
            )

    assert total_users == len(indices)

    return total_loss / total_users

def predict_in_batches(
    model,
    sequences,
    static_inputs,
    labels,
    indices,
    means,
    stds,
):
    model.eval()
    probability_batches = []

    batches = iter_batches(
        sequences,
        static_inputs,
        labels,
        indices,
        means,
        stds,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    with torch.inference_mode():
        for sequence_batch, static_batch, _ in batches:
            logits = model(
                sequence_batch.to(DEVICE),
                static_batch.to(DEVICE),
            )

            cpu_logits = logits.cpu().numpy().astype(np.float64)
            probability_batches.append(expit(cpu_logits))

    probabilities = np.concatenate(probability_batches)

    assert len(probabilities) == len(indices)
    assert np.isfinite(probabilities).all()

    return probabilities

In [9]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

gru_model = ChurnGRU(**model_config).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=LEARNING_RATE,
)

best_valid_loss = np.inf
best_epoch = None
epochs_without_improvement = 0
training_rows = []

valid_labels = np.asarray(
    history_labels[valid_indices],
    dtype=np.int8,
)

for epoch in range(1, MAX_EPOCHS + 1):
    start = perf_counter()

    train_loss = train_one_epoch(
        gru_model,
        optimizer,
        train_indices,
        history_static,
        sequence_means,
        sequence_stds,
        epoch,
    )

    valid_probabilities = predict_in_batches(
        gru_model,
        history_sequences,
        history_static,
        history_labels,
        valid_indices,
        sequence_means,
        sequence_stds,
    )

    valid_loss = log_loss(valid_labels, valid_probabilities)

    assert np.isfinite(valid_loss)

    training_rows.append({
        "epoch": epoch,
        "train_bce": train_loss,
        "valid_log_loss": valid_loss,
        "valid_roc_auc": roc_auc_score(
            valid_labels, valid_probabilities
        ),
        "valid_average_precision": average_precision_score(
            valid_labels, valid_probabilities
        ),
        "epoch_seconds": perf_counter() - start,
    })

    print(
        f"Epoch {epoch} complete | "
        f"train BCE={train_loss:.5f} | "
        f"valid Log Loss={valid_loss:.5f} | "
        f"seconds={training_rows[-1]['epoch_seconds']:.1f}",
        flush=True,
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        cpu_state = {
            name: tensor.detach().cpu()
            for name, tensor in gru_model.state_dict().items()
        }

        torch.save(cpu_state, internal_checkpoint)
    else:
        epochs_without_improvement += 1

    pd.DataFrame(training_rows).to_csv(
        REPORT_DIR / "gru_internal_training_history.csv",
        index=False,
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

gru_model.load_state_dict(
    torch.load(
        internal_checkpoint,
        map_location="cpu",
        weights_only=True,
    )
)

training_history = pd.DataFrame(training_rows)

print("Selected epoch:", best_epoch)
print(f"Best internal validation Log Loss: {best_valid_loss:.6f}")

training_history

Epoch 1 | users=102,400 | mean loss=0.23269
Epoch 1 | users=204,800 | mean loss=0.20925
Epoch 1 | users=307,200 | mean loss=0.19987
Epoch 1 | users=409,600 | mean loss=0.19519
Epoch 1 | users=512,000 | mean loss=0.19222
Epoch 1 | users=614,400 | mean loss=0.18958
Epoch 1 complete | train BCE=0.18852 | valid Log Loss=0.17526 | seconds=23.9
Epoch 2 | users=102,400 | mean loss=0.17900
Epoch 2 | users=204,800 | mean loss=0.17685
Epoch 2 | users=307,200 | mean loss=0.17546
Epoch 2 | users=409,600 | mean loss=0.17483
Epoch 2 | users=512,000 | mean loss=0.17454
Epoch 2 | users=614,400 | mean loss=0.17427
Epoch 2 complete | train BCE=0.17389 | valid Log Loss=0.17017 | seconds=19.2
Epoch 3 | users=102,400 | mean loss=0.16959
Epoch 3 | users=204,800 | mean loss=0.16990
Epoch 3 | users=307,200 | mean loss=0.17085
Epoch 3 | users=409,600 | mean loss=0.17057
Epoch 3 | users=512,000 | mean loss=0.17055
Epoch 3 | users=614,400 | mean loss=0.17019
Epoch 3 complete | train BCE=0.16964 | valid Log Loss=

,epoch,train_bce,valid_log_loss,valid_roc_auc,valid_average_precision,epoch_seconds
0,1,0.188517,0.175259,0.898860,0.705257,23.860131
1,2,0.173892,0.170175,0.906786,0.715719,19.242580
2,3,0.169636,0.166527,0.911842,0.723856,19.778789
3,4,0.166457,0.163509,0.918063,0.737942,19.388004
4,5,0.162561,0.158911,0.923502,0.754236,20.010812
5,6,0.156398,0.150167,0.930579,0.780039,19.301827
6,7,0.148283,0.142335,0.935839,0.796330,19.168643
7,8,0.142138,0.138705,0.938799,0.802797,19.336376
8,9,0.138204,0.135623,0.941902,0.809708,18.701569
9,10,0.136358,0.133816,0.942545,0.811772,19.057153


## Refit on All Historical Users for the Selected Epoch Count

Freeze the epoch count selected on internal historical validation.
Refit preprocessing on all historical users before final training.

In [10]:
from sklearn.base import clone

SELECTED_EPOCHS = int(best_epoch)

final_transaction_preprocessor = clone(transaction_preprocessor)

final_transaction_preprocessor.fit(history_transactions)

history_static_final = final_transaction_preprocessor.transform(
    history_transactions
).astype(np.float32)

assert np.isfinite(history_static_final).all()

final_sequence_means, final_sequence_stds, final_sequence_observations = (
    fit_sequence_normalization(
        history_sequences,
        all_history_indices,
    )
)

final_model_config = {
    **model_config,
    "static_features": history_static_final.shape[1],
    "prior_probability": float(history_labels.mean()),
}

print("Final training users:", len(all_history_indices))
print("Frozen training epochs:", SELECTED_EPOCHS)
print("Final static input dimension:", history_static_final.shape[1])

Final training users: 882127
Frozen training epochs: 12
Final static input dimension: 10


In [11]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

final_gru_model = ChurnGRU(**final_model_config).to(DEVICE)

final_optimizer = torch.optim.Adam(
    final_gru_model.parameters(),
    lr=LEARNING_RATE,
)

final_training_rows = []

for epoch in range(1, SELECTED_EPOCHS + 1):
    start = perf_counter()

    train_loss = train_one_epoch(
        final_gru_model,
        final_optimizer,
        all_history_indices,
        history_static_final,
        final_sequence_means,
        final_sequence_stds,
        epoch,
    )

    final_training_rows.append({
        "epoch": epoch,
        "train_bce": train_loss,
        "epoch_seconds": perf_counter() - start,
    })

    pd.DataFrame(final_training_rows).to_csv(
        REPORT_DIR / "gru_final_training_history.csv",
        index=False,
    )

    print(
        f"Final epoch {epoch}/{SELECTED_EPOCHS} | "
        f"train BCE={train_loss:.5f} | "
        f"seconds={final_training_rows[-1]['epoch_seconds']:.1f}",
        flush=True,
    )

final_cpu_state = {
    name: tensor.detach().cpu()
    for name, tensor in final_gru_model.state_dict().items()
}

torch.save(
    final_cpu_state,
    MODEL_DIR / "gru_final.pt",
)

print("Final historical training complete.")

Epoch 1 | users=102,400 | mean loss=0.23397
Epoch 1 | users=204,800 | mean loss=0.21090
Epoch 1 | users=307,200 | mean loss=0.20087
Epoch 1 | users=409,600 | mean loss=0.19622
Epoch 1 | users=512,000 | mean loss=0.19249
Epoch 1 | users=614,400 | mean loss=0.19022
Epoch 1 | users=716,800 | mean loss=0.18823
Epoch 1 | users=819,200 | mean loss=0.18653
Final epoch 1/12 | train BCE=0.18583 | seconds=21.2
Epoch 2 | users=102,400 | mean loss=0.17448
Epoch 2 | users=204,800 | mean loss=0.17391
Epoch 2 | users=307,200 | mean loss=0.17388
Epoch 2 | users=409,600 | mean loss=0.17325
Epoch 2 | users=512,000 | mean loss=0.17273
Epoch 2 | users=614,400 | mean loss=0.17273
Epoch 2 | users=716,800 | mean loss=0.17239
Epoch 2 | users=819,200 | mean loss=0.17188
Final epoch 2/12 | train BCE=0.17172 | seconds=20.4
Epoch 3 | users=102,400 | mean loss=0.17058
Epoch 3 | users=204,800 | mean loss=0.16939
Epoch 3 | users=307,200 | mean loss=0.16932
Epoch 3 | users=409,600 | mean loss=0.16910
Epoch 3 | users=

In [12]:
import joblib

joblib.dump(
    final_transaction_preprocessor,
    MODEL_DIR / "gru_transaction_preprocessor.joblib",
)

np.savez(
    MODEL_DIR / "gru_sequence_normalization.npz",
    means=final_sequence_means,
    stds=final_sequence_stds,
    observation_counts=final_sequence_observations,
)

gru_training_config = {
    "seed": SEED,
    "device": str(DEVICE),
    "pytorch_version": torch.__version__,
    "model": final_model_config,
    "sequence_channels": sequence_config["sequence_channels"],
    "transaction_columns": sequence_config["transaction_columns"],
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "BCEWithLogitsLoss",
    "gradient_clip_norm": 1.0,
    "selection_max_epochs": MAX_EPOCHS,
    "selection_patience": PATIENCE,
    "selection_stop_reason": (
        "patience"
        if epochs_without_improvement >= PATIENCE
        else "maximum_epoch_budget"
    ),
    "selected_epochs": SELECTED_EPOCHS,
    "best_internal_log_loss": float(best_valid_loss),
    "internal_training_users": len(train_indices),
    "internal_validation_users": len(valid_indices),
    "final_training_users": len(all_history_indices),
    "final_preprocessing_fit_on": "all historical users",
    "later_cohort_used_for_epoch_selection": False,
    "independent_test_evaluation_completed": False,
}

(REPORT_DIR / "gru_training_config.json").write_text(
    json.dumps(gru_training_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Final model preprocessing and configuration saved.")

Final model preprocessing and configuration saved.


## Evaluate the Frozen Model on the Later Cohort

In [13]:
later_sequences = np.load(
    later_dir / "sequences.npy",
    mmap_mode="r",
)

later_transactions = np.load(
    later_dir / "transaction_inputs.npy",
    mmap_mode="r",
)

later_labels = np.load(
    later_dir / "labels.npy",
    mmap_mode="r",
)

later_static_final = final_transaction_preprocessor.transform(
    later_transactions
).astype(np.float32)

assert np.isfinite(later_static_final).all()
assert len(later_labels) == later_sequences.shape[0]
assert len(later_labels) == later_static_final.shape[0]

later_indices = np.arange(len(later_labels))

gru_later_probabilities = predict_in_batches(
    final_gru_model,
    later_sequences,
    later_static_final,
    later_labels,
    later_indices,
    final_sequence_means,
    final_sequence_stds,
)

gru_results = pd.DataFrame([{
    "model": "GRU + transaction inputs",
    "log_loss": log_loss(later_labels, gru_later_probabilities),
    "roc_auc": roc_auc_score(later_labels, gru_later_probabilities),
    "average_precision": average_precision_score(
        later_labels, gru_later_probabilities
    ),
}])

classical_results = pd.read_csv(
    REPORT_DIR / "behavior_model_results.csv"
)

comparison_columns = [
    "model",
    "log_loss",
    "roc_auc",
    "average_precision",
]

classical_combined_results = classical_results.loc[
    classical_results["feature_set"] == "Transaction + listening",
    comparison_columns,
]

final_model_comparison = (
    pd.concat(
        [classical_combined_results, gru_results],
        ignore_index=True,
    )
    .sort_values("log_loss")
    .reset_index(drop=True)
)

final_model_comparison

,model,log_loss,roc_auc,average_precision
0,Logistic regression,0.207310,0.849893,0.550063
1,LightGBM,0.225507,0.839498,0.494052
2,XGBoost,0.230022,0.839235,0.489641
3,GRU + transaction inputs,0.239799,0.809134,0.470220


In [14]:
selected_count = int(np.ceil(len(later_labels) * 0.1))

ranked_indices = np.argsort(
    -gru_later_probabilities,
    kind="stable",
)

selected_indices = ranked_indices[:selected_count]

selected_churns = int(later_labels[selected_indices].sum())
total_churns = int(later_labels.sum())

precision = selected_churns / selected_count

gru_targeting_results = pd.DataFrame([{
    "model": "GRU + transaction inputs",
    "selected_users": selected_count,
    "selected_churned_users": selected_churns,
    "precision_at_10pct": precision,
    "recall_at_10pct": selected_churns / total_churns,
    "lift_at_10pct": precision / float(later_labels.mean()),
    "tie_break": "stable sequence order; msno ascending",
}])

gru_targeting_results

,model,selected_users,selected_churned_users,precision_at_10pct,recall_at_10pct,lift_at_10pct,tie_break
0,GRU + transaction inputs,97096,47199,0.486107,0.540467,5.404672,stable sequence order; msno ascending


In [15]:
gru_results.to_csv(
    REPORT_DIR / "gru_later_results.csv",
    index=False,
)

final_model_comparison.to_csv(
    REPORT_DIR / "final_model_comparison.csv",
    index=False,
)

gru_targeting_results.to_csv(
    REPORT_DIR / "gru_risk_targeting_results.csv",
    index=False,
)

normalization_summary.to_csv(
    REPORT_DIR / "gru_internal_normalization_summary.csv",
    index=False,
)

final_normalization_summary = pd.DataFrame({
    "channel": sequence_config["sequence_channels"][:7],
    "training_observations": final_sequence_observations,
    "mean": final_sequence_means,
    "std": final_sequence_stds,
})

final_normalization_summary.to_csv(
    REPORT_DIR / "gru_final_normalization_summary.csv",
    index=False,
)

print("GRU evaluation and normalization reports saved.")

GRU evaluation and normalization reports saved.


## PyTorch GRU Benchmark

Built a GRU-based model using 30-day daily listening sequences and transaction inputs. Sequence masks distinguish missing log days from recorded days with invalid durations.

Epoch selection used an internal historical split. The best internal validation Log Loss was 0.132391 at epoch 12. The run reached its predefined 12-epoch budget; patience-based early stopping did not trigger.

The final model and preprocessing were refitted on all 882,127 historical users for 12 epochs, then evaluated on the same 970,960-user later cohort used for the classical models.

| Model | Later Log Loss ↓ | ROC-AUC ↑ | Average Precision ↑ |
|---|---:|---:|---:|
| Logistic Regression | **0.207310** | **0.849893** | **0.550063** |
| LightGBM | 0.225507 | 0.839498 | 0.494052 |
| XGBoost | 0.230022 | 0.839235 | 0.489641 |
| GRU + transaction inputs | 0.239799 | 0.809134 | 0.470220 |

The GRU did not outperform the classical baselines in this evaluation. Strong internal historical performance did not carry over to the later cohort.

The comparison uses matching cohorts, source data, and feature cutoffs, but different input representations. It does not isolate the effect of sequence order or architecture. Previously documented label-provenance and retrospective-evaluation limitations remain applicable.

Logistic Regression is the strongest benchmark observed so far; deployment suitability and intervention effectiveness have not been established.

See [the GRU notebook](notebooks/16_gru_model.ipynb), [training configuration](reports/gru_training_config.json), and [model comparison](reports/final_model_comparison.csv).